# Smart Inventory AI Assistant
by: Nadia Chusnul Ikromah

Github: [https://github.com/nadyacii](https://github.com/nadyacii)

Linkedin: [www.linkedin.com/in/nadiacikr](www.linkedin.com/in/nadiacikr)

## Apa itu Smart Inventory AI Assistant?
Smart Inventory AI Assistant adalah chatbot berbasis Artificial Intelligence yang membantu pengguna dalam melakukan analisis inventory secara otomatis. Aplikasi ini memanfaatkan Google Gemini API sebagai Large Language Model (LLM) dan Streamlit sebagai antarmuka pengguna (User Interface).

## Step 1 — Install Required Libraries

Pada tahap ini, kita akan menginstal seluruh library yang dibutuhkan untuk membangun aplikasi **Smart Inventory AI Assistant**.

Library yang digunakan memiliki fungsi sebagai berikut:

| Library           | Fungsi                                                                             |
| ----------------- | ---------------------------------------------------------------------------------- |
| **streamlit**     | Membangun antarmuka web interaktif untuk chatbot dan dashboard inventory           |
| **google-genai**  | Menghubungkan aplikasi dengan Google Gemini API sebagai Large Language Model (LLM) |
| **pandas**        | Membaca, memproses, dan menganalisis dataset inventory                             |
| **plotly**        | Membuat visualisasi data yang interaktif                                           |
| **python-dotenv** | Mengelola API Key dan konfigurasi lingkungan aplikasi                              |
| **openpyxl**      | Membaca dan memproses file Excel (.xlsx)                                           |
| **pyngrok**       | Membuat URL publik sehingga aplikasi Streamlit dapat diakses melalui browser       |

Jalankan cell berikut untuk menginstal seluruh dependensi yang diperlukan sebelum melanjutkan ke tahap berikutnya.


In [55]:
# Install required libraries
!pip install -q streamlit
!pip install -q google-genai
!pip install -q pandas
!pip install -q plotly
!pip install -q python-dotenv
!pip install -q openpyxl
!pip install -q pyngrok

## Step 2 — Load and Explore the Dataset

Dataset inventory akan diunggah menggunakan library Pandas. Dataset yang digunakan berisi informasi terkait kondisi inventory, penjualan, dan prediksi permintaan produk yang akan digunakan sebagai sumber data utama dalam proses analisis dan rekomendasi AI.

Dataset ini dapat diunduh melalui Kaggle:

Kaggle Dataset:
https://www.kaggle.com/code/jijagallery/retail-store-inventory-forecasting?select=retail_store_inventory.csv

In [56]:
# Load the dataset and preview the data
import pandas as pd

df = pd.read_csv(
    "retail_store_inventory.csv"
)

df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


## Step 3 — Select Relevant Features for Inventory Analysis

Setelah dataset berhasil dimuat, langkah berikutnya adalah melakukan **data preparation** dengan memilih kolom-kolom yang relevan untuk analisis inventory.

Selanjutnya, hanya beberapa kolom yang dipilih karena memiliki peran penting dalam pengelolaan inventory, yaitu:

| Column              | Description                                  |
| ------------------- | -------------------------------------------- |
| **Store ID**        | Identitas toko atau cabang                   |
| **Product ID**      | Identitas unik produk                        |
| **Category**        | Kategori produk                              |
| **Inventory Level** | Jumlah stok yang tersedia saat ini           |
| **Units Sold**      | Jumlah produk yang terjual                   |
| **Demand Forecast** | Prediksi permintaan produk di masa mendatang |

Pemilihan fitur ini bertujuan untuk menyederhanakan proses analisis dan memastikan bahwa chatbot hanya menggunakan informasi yang relevan untuk menghasilkan rekomendasi inventory yang lebih akurat.


In [57]:
# Create a copy of the dataset and select relevant columns for inventory analysis
analysis_df = df.copy()

analysis_df = analysis_df[
    [
        "Store ID",
        "Product ID",
        "Category",
        "Inventory Level",
        "Units Sold",
        "Demand Forecast"
    ]
]

## Step 4 — Calculate Inventory Coverage

Selanjutnya, dilakukan perhitungan **Inventory Coverage** untuk mengukur kecukupan stok terhadap prediksi permintaan (*Demand Forecast*).

Formula yang digunakan:

**Inventory Coverage = Inventory Level ÷ Demand Forecast**

Nilai Inventory Coverage akan digunakan sebagai indikator utama untuk mengidentifikasi produk yang berisiko mengalami **stockout**, memiliki kondisi **normal**, atau mengalami **overstock** pada tahap analisis berikutnya.

In [58]:
# Calculate inventory coverage based on inventory level and demand forecast
analysis_df["Inventory_Coverage"] = (
    analysis_df["Inventory Level"] /
    analysis_df["Demand Forecast"]
)

Setelah nilai Inventory Coverage dihitung, langkah berikutnya adalah mengklasifikasikan kondisi inventory setiap produk.

Klasifikasi dilakukan berdasarkan aturan berikut:

- Stockout → Inventory Coverage < 1
- Normal → Inventory Coverage antara 1 dan 5
- Overstock → Inventory Coverage > 5

Fungsi ini akan digunakan untuk memberikan label kondisi inventory pada setiap produk sehingga memudahkan proses monitoring dan pengambilan keputusan terkait restock maupun pengurangan stok.

In [59]:
# Define a function to classify inventory status based on coverage ratio
def classify_inventory(coverage):

    if coverage < 1:
        return "Stockout"

    elif coverage > 5:
        return "Overstock"

    else:
        return "Normal"

Pada tahap ini, data inventory diurutkan berdasarkan nilai **Inventory Coverage** dari yang terendah hingga tertinggi.

Produk dengan nilai Inventory Coverage paling rendah memiliki risiko kehabisan stok (*stockout*) yang lebih tinggi, sehingga perlu diprioritaskan untuk dilakukan restock.


In [60]:
# Sort products by inventory coverage to identify restocking priorities
priority_restock = (
    analysis_df
    .sort_values(
        by="Inventory_Coverage"
    )
)

Selanjutnya, sistem memilih 20 produk dengan risiko tertinggi berdasarkan hasil analisis inventory sebelumnya.

Data tersebut kemudian dikonversi menjadi format teks menggunakan fungsi `to_string()`. Tujuannya adalah agar informasi inventory dapat dikirim sebagai context ke Google Gemini API.

Dengan memberikan data produk yang paling berisiko sebagai context, chatbot dapat menghasilkan rekomendasi yang lebih relevan dan akurat, seperti produk yang perlu segera direstock atau kategori yang memiliki risiko stockout tertinggi.

In [61]:
# Select the top 20 high-risk products and convert them to text for AI context
top_risk = priority_restock.head(20)

inventory_context = top_risk.to_string()

## Step 5 — Create the AI Prompt

Sebuah pertanyaan dibuat untuk diajukan kepada AI serta *prompt* yang berisi konteks inventory dan instruksi chatbot.

Prompt berfungsi sebagai panduan bagi Google Gemini agar memahami perannya sebagai **Smart Inventory AI Assistant**, menggunakan data inventory yang telah dianalisis sebelumnya, serta memberikan jawaban dalam Bahasa Indonesia.

In [63]:
# Set the inventory management question for AI analysis
question = """
Produk mana yang harus segera direstock?
"""

In [64]:
# Build the prompt context
chat_prompt = f"""
Anda adalah Smart Inventory AI Assistant.

Selalu jawab menggunakan Bahasa Indonesia.

Data Inventory:

{inventory_context}

Pertanyaan:

{question}
"""

## Step 6 — Configure the Gemini API Client

### Google AI API Key

Cara mendapatkan:
1. Buka [https://aistudio.google.com](https://aistudio.google.com)
2. Klik **Get API Key** → **Create API Key**
3. Di Google Colab: klik ikon 🔑 (Secrets) di sidebar kiri → tambahkan secret `GEMINI`

Konfigurasi dan autentikasi Google Gemini API digunakan sebagai mesin AI pada aplikasi Smart Inventory AI Assistant.

API Key disimpan secara aman menggunakan Google Colab Secrets (userdata) sehingga tidak perlu dituliskan langsung di dalam kode. Setelah API Key berhasil diambil, sistem akan membuat objek Gemini Client yang berfungsi sebagai penghubung antara aplikasi dan model Gemini.

In [65]:
# Configure and authenticate the Gemini API client
from google import genai
from google.colab import userdata

GEMINI = userdata.get('GEMINI')

client = genai.Client(
    api_key=GEMINI
)

## Step 7 — Generate AI Recommendations

Prompt yang telah dibuat sebelumnya dikirim ke **Google Gemini API** untuk diproses oleh model AI.

Model Gemini akan menganalisis data inventory yang diberikan sebagai konteks, memahami pertanyaan pengguna, kemudian menghasilkan rekomendasi dan insight berdasarkan kondisi inventory yang tersedia.

In [66]:
# Send the prompt to Gemini and display the AI response
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=chat_prompt
)

print(response.text)

Berdasarkan data yang Anda berikan, kami akan mengidentifikasi produk yang harus segera direstock dengan memprioritaskan nilai `Inventory_Coverage` yang paling rendah (paling negatif), karena ini menunjukkan cakupan inventaris yang paling kritis atau kurang.

Berikut adalah produk yang harus segera direstock, diurutkan dari yang paling mendesak:

1.  **Product ID: P0005** (Store ID: S001, Category: Toys)
    *   Inventory Level: 444
    *   Units Sold: 3
    *   Demand Forecast: -0.05
    *   **Inventory_Coverage: -8880.00**
    *   *Keterangan: Memiliki nilai Inventory_Coverage paling rendah, menandakan kondisi stok paling kritis.*

2.  **Product ID: P0017** (Store ID: S005, Category: Groceries)
    *   Inventory Level: 331
    *   Units Sold: 1
    *   Demand Forecast: -0.04
    *   **Inventory_Coverage: -8275.00**

3.  **Product ID: P0002** (Store ID: S004, Category: Clothing)
    *   Inventory Level: 68
    *   Units Sold: 1
    *   Demand Forecast: -0.01
    *   **Inventory_Covera

## Step 8 — Configure Ngrok Authentication

Selanjutnya, dilakukan konfigurasi **Ngrok Authentication Token** yang digunakan untuk membuat URL publik bagi aplikasi Streamlit yang berjalan di Google Colab.

Karena aplikasi Streamlit dijalankan pada lingkungan lokal Colab, aplikasi tersebut tidak dapat diakses langsung melalui internet. Oleh karena itu, **Ngrok** digunakan untuk membuat *secure tunnel* yang menghubungkan aplikasi lokal dengan URL publik yang dapat diakses melalui browser.

Cara mendapatkan token:
1. Daftar gratis di [https://ngrok.com](https://ngrok.com)
2. Login → klik **Your Authtoken** di dashboard
3. Copy token-nya, paste ke Colab Secrets dengan nama `NGROK_TOKEN`
   (klik ikon 🔑 di sidebar kiri Colab)

- Token ngrok gratis sudah cukup untuk keperluan training.
- Satu akun ngrok bisa membuka 1 tunnel aktif sekaligus.


In [67]:
# Set up ngrok authentication for public access
from pyngrok import ngrok
from google.colab import userdata

# Retrieve the ngrok token from Colab Secrets
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
print("ngrok token berhasil dikonfigurasi!")

ngrok token berhasil dikonfigurasi!


## Step 9 — Launch the Streamlit Application

Dibuat sebuah fungsi untuk menjalankan aplikasi **Streamlit** secara otomatis dan menghasilkan URL publik menggunakan **Ngrok**.

Fungsi `run_streamlit()` memiliki beberapa tugas utama:

1. Menghentikan proses Streamlit yang masih berjalan dari sesi sebelumnya.
2. Membersihkan port yang sedang digunakan agar tidak terjadi konflik.
3. Menutup tunnel Ngrok yang masih aktif.
4. Menjalankan aplikasi Streamlit secara *headless* di Google Colab.
5. Membuat URL publik menggunakan Ngrok sehingga aplikasi dapat diakses melalui browser.

Dengan fungsi ini, proses deployment aplikasi menjadi lebih sederhana karena pengguna hanya perlu memanggil fungsi `run_streamlit()` untuk menjalankan aplikasi dan mendapatkan URL publik secara otomatis.

In [68]:
# Run a Streamlit app and generate a public ngrok URL
import subprocess
import time

def run_streamlit(filename, port=8501):
    # Clean up existing Streamlit processes and tunnels
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)
    ngrok.kill()

    time.sleep(3)

    # Launch Streamlit
    proc = subprocess.Popen(
        [
            "streamlit", "run", filename,
            "--server.headless=true",
            "--server.port", str(port),
            "--server.enableCORS=false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(3)

    # Expose the app through ngrok
    public_url = ngrok.connect(port)
    print(f"Streamlit berjalan: {public_url}")

    return proc

## Step 10 — Build the Streamlit Application

Pada tahap ini, seluruh komponen aplikasi **Smart Inventory AI Assistant** diintegrasikan ke dalam file `app.py` menggunakan framework **Streamlit**.

Aplikasi ini menggabungkan berbagai fitur yang telah dibuat sebelumnya, mulai dari proses upload dataset, analisis inventory, visualisasi data, hingga integrasi chatbot berbasis Google Gemini.

Fitur utama yang tersedia dalam aplikasi meliputi:

* 📂 Upload dataset inventory dalam format CSV atau Excel.
* 📄 Preview dataset yang diunggah pengguna.
* 📊 Dashboard inventory untuk memantau kondisi stok.
* 📈 Visualisasi data menggunakan grafik interaktif Plotly.
* 🚨 Identifikasi produk yang perlu diprioritaskan untuk restock.
* 🤖 Chatbot AI berbasis Gemini untuk memberikan insight dan rekomendasi inventory.

File `app.py` berfungsi sebagai inti aplikasi yang akan dijalankan melalui Streamlit dan diakses melalui URL publik yang dibuat menggunakan Ngrok.


In [69]:
# Import required libraries for the Streamlit
%%writefile app.py

import streamlit as st
import pandas as pd
from google import genai
import plotly.express as px
import plotly.graph_objects as go


# GEMINI CONFIG
GEMINI = "YOUR_API_KEY"

client = genai.Client(api_key=GEMINI)


#CHAT MEMORY
if "messages" not in st.session_state:
    st.session_state.messages = []


# AI FUNCTION
def ask_inventory_ai(question, inventory_context):
    prompt = f"""
    Anda adalah Smart Inventory AI Assistant.

    Keahlian:
    - Inventory Management
    - Supply Chain Management
    - Warehouse Management
    - Stock Optimization

    Aturan:
    - Selalu jawab dalam Bahasa Indonesia.
    - Gunakan bahasa yang profesional dan mudah dipahami.
    - Berikan rekomendasi yang praktis.
    - Jelaskan alasan dari setiap rekomendasi.

    Data Inventory:

    {inventory_context}

    Pertanyaan Pengguna:

    {question}
    """
    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt
    )
    return response.text


#PAGE CONFIG
st.set_page_config(
    page_title="Smart Inventory AI Assistant",
    page_icon="📦",
    layout="wide"
)

st.title("📦 Smart Inventory AI Assistant")
st.markdown("AI-Powered Inventory Management Chatbot using Gemini API.")
st.divider()


# FILE UPLOAD
uploaded_file = st.file_uploader(
    "📂 Upload Dataset Inventory",
    type=["csv", "xlsx", "xls"]
)


# MAIN APP
if uploaded_file:

    df = pd.read_csv(uploaded_file)
    st.success("✅ Dataset berhasil diupload!")

    col_title, col_input = st.columns([3, 1])

    with col_title:
        st.subheader("Preview Dataset")

    with col_input:
        preview_rows = st.number_input(
            "Jumlah Baris",
            min_value=1,
            max_value=len(df),
            value=5,
            step=1
        )

    st.dataframe(df.head(preview_rows))


    # INVENTORY ANALYSIS
    analysis_df = df.copy()
    analysis_df["Inventory_Coverage"] = (
        analysis_df["Inventory Level"] / analysis_df["Demand Forecast"]
    )

    def classify_inventory(coverage):
        if coverage < 1:
            return "Stockout Risk"
        elif coverage > 5:
            return "Overstock Risk"
        else:
            return "Normal"

    analysis_df["Status"] = analysis_df["Inventory_Coverage"].apply(classify_inventory)


    #DASHBOARD
    stockout  = (analysis_df["Status"] == "Stockout Risk").sum()
    normal    = (analysis_df["Status"] == "Normal").sum()
    overstock = (analysis_df["Status"] == "Overstock Risk").sum()


    #KPI
    st.subheader("📊 Inventory Dashboard")
    col1, col2, col3 = st.columns(3)
    col1.metric("⚠️ Stockout Risk", stockout)
    col2.metric("✅ Normal", normal)
    col3.metric("📦 Overstock Risk", overstock)

    st.divider()


    # CHARTS
    CHART_HEIGHT = 350

    BORDER_SHAPE = dict(
        type="rect",
        xref="paper", yref="paper",
        x0=0, y0=0, x1=1, y1=1,
        line=dict(color="black", width=2)
    )

    BORDER_STYLE = dict(
        showlegend=True,
        margin=dict(l=40, r=20, t=50, b=50),
        paper_bgcolor="white",
        plot_bgcolor="white",
        height=CHART_HEIGHT
    )

    chart_col1, chart_col2, chart_col3 = st.columns(3)


    # LINE CHART
    sales_trend = (
        analysis_df
        .groupby("Date")["Units Sold"]
        .sum()
        .reset_index()
    )

    fig_line = go.Figure()

    fig_line.add_trace(go.Scatter(
        x=sales_trend["Date"],
        y=sales_trend["Units Sold"],
        mode="lines+markers",
        line=dict(color="#2563EB", width=2),
        marker=dict(size=4),
        name="Units Sold"
    ))

    fig_line.update_layout(
        title=dict(text="📈 Tren Penjualan", x=0.5, xanchor="center"),
        xaxis_title="Tanggal",
        yaxis_title="Units Sold",
        shapes=[BORDER_SHAPE],
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        **BORDER_STYLE
    )

    with chart_col1:
        st.plotly_chart(fig_line, use_container_width=True)


    # BAR CHART
    sales_category = (
        analysis_df
        .groupby("Category")["Units Sold"]
        .sum()
        .reset_index()
        .sort_values("Units Sold", ascending=False)
    )

    n_bars = len(sales_category)
    blue_gradient = [
        f"rgb({int(10 + (180/max(n_bars-1,1)) * i)}, "
        f"{int(80 + (120/max(n_bars-1,1)) * i)}, "
        f"{int(140 + (115/max(n_bars-1,1)) * i)})"
        for i in range(n_bars - 1, -1, -1)
    ]

    fig_bar = go.Figure()

    fig_bar.add_trace(go.Bar(
        x=sales_category["Category"],
        y=sales_category["Units Sold"],
        marker_color=blue_gradient,
        width=0.6,
        text=sales_category["Units Sold"],
        textposition="outside",
        textfont=dict(size=11, color="black"),
        name="Units Sold"
    ))

    fig_bar.update_layout(
        title=dict(text="📊 Penjualan per Kategori", x=0.5, xanchor="center"),
        xaxis_title="Kategori",
        yaxis_title="Units Sold",
        yaxis=dict(range=[0, sales_category["Units Sold"].max() * 1.25]),
        shapes=[BORDER_SHAPE],
        showlegend=False,
        margin=dict(l=40, r=20, t=50, b=50),
        paper_bgcolor="white",
        plot_bgcolor="white",
        height=CHART_HEIGHT
    )

    with chart_col2:
        st.plotly_chart(fig_bar, use_container_width=True)


    # DONUT CHART
    donut_labels = ["Stockout", "Normal", "Overstock"]
    donut_values = [stockout, normal, overstock]

    donut_data = sorted(
        zip(donut_values, donut_labels),
        reverse=True
    )
    sorted_values = [v for v, _ in donut_data]
    sorted_labels = [l for _, l in donut_data]

    donut_blue_gradient = [
        "rgb(10, 80, 140)",
        "rgb(70, 140, 200)",
        "rgb(160, 210, 255)",
    ]

    fig_donut = go.Figure()

    fig_donut.add_trace(go.Pie(
        labels=sorted_labels,
        values=sorted_values,
        hole=0.5,
        marker=dict(
            colors=donut_blue_gradient,
            line=dict(color="white", width=2)
        ),
        textinfo="percent",
        textfont=dict(size=11),
        domain=dict(x=[0.05, 0.95], y=[0.05, 0.95]),
        name=""
    ))

    fig_donut.update_layout(
        title=dict(text="🍩 Status Inventory", x=0.5, xanchor="center"),
        shapes=[BORDER_SHAPE],
        legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="right", x=1.15),
        **BORDER_STYLE
    )

    with chart_col3:
        st.plotly_chart(fig_donut, use_container_width=True)


    # PRIORITY RESTOCK
    priority_restock = analysis_df.sort_values(by="Inventory_Coverage")
    top_risk = priority_restock.head(10)

    st.subheader("🚨 Top 10 Priority Restock")
    st.dataframe(
        top_risk[[
            "Product ID", "Category", "Inventory Level",
            "Demand Forecast", "Inventory_Coverage", "Status"
        ]]
    )


    #INVENTORY CONTEXT FOR GEMINI
    inventory_context = f"""
    Inventory Summary

    Total Products: {len(analysis_df)}
    Stockout Risk: {stockout}
    Normal: {normal}
    Overstock Risk: {overstock}

    Sample Inventory Data:
    {analysis_df.head(20).to_string()}
    """

    st.divider()


    # CHATBOT
    st.subheader("🤖 Inventory AI Chat")

    # Build container for history and input
    chat_container = st.container()
    question = st.chat_input("Tanyakan sesuatu tentang inventory Anda...")

    # Process the input
    if question:
        st.session_state.messages.append({"role": "user", "content": question})
        answer = ask_inventory_ai(question, inventory_context)
        st.session_state.messages.append({"role": "assistant", "content": answer})

    # Render history only in chat_container
    with chat_container:
        for message in st.session_state.messages:
            with st.chat_message(message["role"]):
                st.write(message["content"])

Overwriting app.py


## Step 11 — Run the Streamlit Application

Pada tahap terakhir, fungsi `run_streamlit()` dijalankan untuk memulai aplikasi **Smart Inventory AI Assistant**.

Fungsi ini akan secara otomatis:

* Menjalankan file `app.py` menggunakan Streamlit.
* Membuat *secure tunnel* menggunakan Ngrok.
* Menghasilkan URL publik yang dapat diakses melalui browser.

Setelah cell ini dijalankan, sistem akan menampilkan URL Ngrok yang menjadi alamat aplikasi. Pengguna dapat membuka URL tersebut untuk mulai menggunakan Smart Inventory AI Assistant, mengunggah dataset inventory, melihat dashboard analisis, serta berinteraksi dengan chatbot AI.

Jika proses berhasil, akan muncul output seperti berikut:

```text
Streamlit berjalan:
https://xxxxxxxx.ngrok-free.app
```

Klik URL yang dihasilkan untuk membuka aplikasi dan mulai melakukan analisis inventory menggunakan AI.


In [70]:
# Launch the Streamlit application and create a public access URL
proc = run_streamlit("app.py")

Streamlit berjalan: NgrokTunnel: "https://shun-platter-untitled.ngrok-free.dev" -> "http://localhost:8501"


## Step 15 — Stop the Streamlit Application

Setelah proses pengujian atau demonstrasi selesai, aplikasi Streamlit dapat dihentikan untuk membebaskan resource yang digunakan pada Google Colab.

Kode berikut digunakan untuk menghentikan proses Streamlit yang sedang berjalan dengan aman menggunakan method `terminate()` pada objek `proc` yang sebelumnya dibuat saat menjalankan aplikasi.

Jika aplikasi berhasil dihentikan, sistem akan menampilkan pesan:

```text
Aplikasi dihentikan.
```

Langkah ini bersifat opsional, namun disarankan untuk dilakukan setelah selesai menggunakan aplikasi agar tidak ada proses yang tetap berjalan di background.


In [71]:
# Stop the running Streamlit application safely
import os, signal
try:
    proc.terminate()
    print("Aplikasi dihentikan.")
except:
    pass

Aplikasi dihentikan.


## Project Conclusion

Selamat! Anda telah berhasil membangun **Smart Inventory AI Assistant**, sebuah aplikasi berbasis Artificial Intelligence yang menggabungkan analisis inventory, visualisasi data, dan chatbot berbasis Large Language Model (LLM) menggunakan Google Gemini API.

Melalui project ini, beberapa konsep penting telah berhasil diimplementasikan, antara lain:

* Data loading dan preprocessing menggunakan Pandas.
* Perhitungan Inventory Coverage untuk mengukur kecukupan stok.
* Klasifikasi kondisi inventory menjadi Stockout, Normal, dan Overstock.
* Identifikasi produk dengan prioritas restock tertinggi.
* Integrasi Google Gemini API untuk menghasilkan rekomendasi berbasis AI.
* Pembuatan dashboard interaktif menggunakan Streamlit dan Plotly.
* Deployment aplikasi menggunakan Ngrok agar dapat diakses melalui browser.

Project ini menunjukkan bagaimana teknologi Artificial Intelligence dapat dimanfaatkan untuk membantu proses pengambilan keputusan dalam manajemen inventory secara lebih cepat, efisien, dan berbasis data.

Terima kasih telah mengikuti implementasi project ini. Semoga project **Smart Inventory AI Assistant** dapat menjadi dasar untuk pengembangan solusi AI yang lebih kompleks di masa mendatang.

🚀 Happy Learning and Keep Building with AI!
